loading in necessary repos

In [1]:
import pandas as pd
import nibabel as nib
import numpy as np
import yaml
import os
import sys
import argparse
import glob

from sklearn.model_selection import train_test_split

#### Data Preparation
obtaining a list of scan ids from data folder 

In [2]:
# COMMENT: FOR NOW, WE'RE WORKING WITH ONLY LEFT HEMISPHERE BRAIN DATA.
hemi = "LEFT"

ico_res = 6
ico_down_res = 2
down_num_vertices = 153 
down_num_patches = 320

## destination privacy etc? ask janine

patch_extraction_path = "/ceph/chpc/shared/janine_bijsterbosch_group/lucas_scratch/SiT_structural_encoder/patch_extraction"
data_path = "/ceph/chpc/shared/janine_bijsterbosch_group/lucas_scratch/HCP_YA"

scans_ids = np.array([f for f in os.listdir(f"{data_path}/resampled_icos") if f.isnumeric()])
print(len(scans_ids))

1113


In [3]:
# COMMENT: this is just other metadata stuff we need to be able to process brain scans and sort

indices_mesh_triangles=pd.read_csv(f'{patch_extraction_path}/triangle_indices_ico_{ico_res}_sub_ico_{ico_down_res}.csv')
id_label_info = pd.read_csv(f"{data_path}/unrestricted_hcp_freesurfer.csv").to_numpy()[:, 0:2]
restricted_access_data = pd.read_csv(f"{data_path}/HCP_YA_subjects_2026_10_22_12_06_42.csv")

obtaining a matrix linking family id with subject id

In [7]:
# COMMENT: [0, 5, 6, 7] --> [subject_id, family_id, mother_id, father_id]
family_info = restricted_access_data.iloc[:, [0, 5,6,7]].to_numpy()

In [ ]:
def preprocess_and_sort(ids, family_ids, train_split, test_split):
    # id_label_info -> [subject_id, subject_sex]
    # redundant, but makes lookup easier
    sex_dict = {str(row[0]): row[1] for row in id_label_info}
    result_dict = {'train':[], 'test':[], 'validation':[]}

    # split on families
    train_family_ids, temp = train_test_split(np.unique(family_ids), train_size=train_split) 
    test_family_ids, val_family_ids = train_test_split(temp, train_size=test_split/(1-train_split))
    
    for i in ids:
        # return the family_id corresponding to the subject id, and bins the subject id in the appropriate bucket
        fam_id = family_info[family_info[: , 0] == int(i), 1]
        if fam_id in train_family_ids:
            result_dict['train'].append(int(i))
        if fam_id in test_family_ids:
            result_dict['test'].append(int(i))
        if fam_id in val_family_ids:
            result_dict['validation'].append(int(i))

    print("train size: ", len(result_dict['train']))
    print("test size: ", len(result_dict['test']))
    print("validation size: ", len(result_dict['validation']))


    for split in ['train', 'test', 'validation']:
        # creates and saves the matrix for train/test/val
        selected_ids = result_dict[split]
        num_subjects = len(selected_ids)
        data = np.zeros((num_subjects, 1, down_num_patches, down_num_vertices)) # 1 == number of channels
        labels = np.zeros((num_subjects, 1))
        subject_ids = np.zeros((num_subjects, 1))

        for i, idx in enumerate(selected_ids):
            # boolean label for subject sex
            subject_sex = int(sex_dict[str(idx)] == "F")
            
            labels[i] = subject_sex
            subject_ids[i] = idx
            
            '''
            labels[i+num_subjects] = subject_sex
            '''

            # load in the left side data from the HCPYA database we have on site
            L = f"{data_path}/resampled_icos/{idx}/{idx}_ico{ico_res}.L.shape.gii"
            loaded_L_hemi = np.array(nib.load(L).agg_data()) # in the future, w/ more channels we can change this
        
            '''
            R = f"{data_path}/resampled_icos/{idx}/{idx}_ico{ico_res}.R.shape.gii"
            loaded_R_hemi = np.array(nib.load(R).agg_data())
            '''

            ## downsampling from ico6 to ico2 using the extraction file
            for j in range(down_num_patches):
            
                indices_to_extract = indices_mesh_triangles[str(j)].to_numpy()
                data[i,:,j,:] = loaded_L_hemi[indices_to_extract]
                '''
                data[i+num_subjects,:,j,:] = loaded_R_hemi[indices_to_extract]
                '''

        result_dict[split] = (data, labels, subject_ids)
    
    return result_dict

In [6]:
family_data = np.unique(family_info[:, 1])
data_package = preprocess_and_sort(scans_ids, family_data, 0.7, 0.15)

train size:  772
test size:  167
validation size:  174


Save processed data

In [12]:
for i in ['train', 'test', 'validation']:
    data_file = f"{data_path}/{i}_L_data.npy"
    label_file = f"{data_path}/{i}_L_labels.npy"
    subjects_file = f"{data_path}/{i}_L_subjects.npy"

    np.save(data_file, data_package[i][0])
    np.save(label_file, data_package[i][1])
    np.save(subjects_file, data_package[i][2])
